In [7]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser  


In [3]:
##reading the pdfs from the folder
loader = DirectoryLoader(
    path="./pdfs",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    # recursive=True,
)
documents = loader.load()
print(len(documents))

63


In [4]:
##splitting the pdfs into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 250)
final_doc = text_splitter.split_documents(documents)
final_doc[:4]
len(final_doc)

331

In [16]:
from langchain_core.prompts import ChatPromptTemplate
##creating embeddigns
hugging_embed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                      model_kwargs={"device": "cpu"},
                                      encode_kwargs={"normalize_embeddings": True}
                                      )

##vector store
vector_store = FAISS.from_documents(final_doc,hugging_embed)
##
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

##creating prompt template
prompt = ChatPromptTemplate.from_template("""Use the following pieces of context to answer the users question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
context:{context}
question:{question}
""")



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5772.11it/s]


In [18]:
from langchain_groq import ChatGroq
def context(docs):
    return"\n\n".join(doc.page_content for doc in docs)

llm = ChatGroq(model = "llama-3.1-8b-instant", temperature = 0.5)

In [22]:
doc_chain = ( prompt 
             | llm 
             | StrOutputParser()
             )
rag_chain = {
    "context": retriever | RunnableLambda(context),
    "question" : RunnablePassthrough(),
}|doc_chain

res = rag_chain.invoke("MEDIAN HOUSEHOLD INCOME?")
res

'The median household income in the United States in 2022 was $74,755, according to Figure 1.'